# Fragrantica Data Pipeline
**Project:** Fragheadsunited — AI Fragrance Recommendation Engine  
**Doel:** Data inladen, combineren, opschonen en voorbereiden voor NLP embeddings

---

## 0. Installatie & Imports

In [ ]:
# Installeer benodigde packages (eenmalig)
# !pip install pandas numpy sentence-transformers scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import ast
import re
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 100)
print('Imports succesvol geladen.')

---
## 1. Data Inladen

Twee datasets:
- `fragrantica_cleaned.csv` — gestructureerde data (noten, accords, ratings)
- `fragrantica_data.csv` — ruwe data met reviews en beschrijvingen

In [ ]:
# ============================================================
# AANPASSEN: zet hier de juiste bestandspaden
PATH_CLEANED = 'fragrantica_cleaned.csv'
PATH_RAW     = 'fragrantica_data.csv'
# ============================================================

df_clean = pd.read_csv(PATH_CLEANED)
df_raw   = pd.read_csv(PATH_RAW)

print(f'Cleaned dataset: {df_clean.shape[0]:,} rijen, {df_clean.shape[1]} kolommen')
print(f'Raw dataset:     {df_raw.shape[0]:,} rijen, {df_raw.shape[1]} kolommen')

In [ ]:
# Eerste blik op cleaned dataset
df_clean.head(3)

In [ ]:
# Eerste blik op raw dataset (met reviews)
df_raw.head(3)

In [ ]:
# Kolomoverzicht
print('=== CLEANED KOLOMMEN ===')
print(df_clean.columns.tolist())
print()
print('=== RAW KOLOMMEN ===')
print(df_raw.columns.tolist())

---
## 2. Data Verkenning (EDA)

In [ ]:
# Missing values overzicht
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

miss_clean = (df_clean.isnull().sum() / len(df_clean) * 100).sort_values(ascending=False)
miss_raw   = (df_raw.isnull().sum()   / len(df_raw)   * 100).sort_values(ascending=False)

miss_clean[miss_clean > 0].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Missing values (%) — Cleaned dataset')
axes[0].set_ylabel('% Missing')

miss_raw[miss_raw > 0].plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Missing values (%) — Raw dataset')

plt.tight_layout()
plt.show()

In [ ]:
# Rating verdeling
if 'Rating Value' in df_clean.columns:
    plt.figure(figsize=(8, 4))
    df_clean['Rating Value'].dropna().plot(kind='hist', bins=30, color='steelblue', edgecolor='white')
    plt.title('Verdeling van ratings')
    plt.xlabel('Rating')
    plt.ylabel('Aantal parfums')
    plt.show()
    print(df_clean['Rating Value'].describe())

In [ ]:
# Top 15 merken
brand_col = 'Brand' if 'Brand' in df_clean.columns else 'designer'
top_brands = df_clean[brand_col].value_counts().head(15)

plt.figure(figsize=(10, 5))
top_brands.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Top 15 meest voorkomende merken')
plt.ylabel('Aantal parfums')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Gender verdeling
if 'Gender' in df_clean.columns:
    gender_counts = df_clean['Gender'].value_counts()
    plt.figure(figsize=(6, 4))
    gender_counts.plot(kind='bar', color=['steelblue', 'coral', 'mediumseagreen'])
    plt.title('Verdeling gender-categorieën')
    plt.ylabel('Aantal parfums')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

---
## 3. Reviews Parsen (Raw Dataset)

Reviews zijn opgeslagen als string-representatie van een Python lijst, bijv. `["review 1", "review 2"]`

In [ ]:
def parse_list_column(val):
    """Zet string-lijst om naar echte Python list."""
    if pd.isna(val) or val == '[]':
        return []
    try:
        result = ast.literal_eval(val)
        return result if isinstance(result, list) else [str(result)]
    except:
        return [str(val)]


def clean_text(text):
    """Basis tekstopschoning."""
    if not isinstance(text, str):
        return ''
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)       # dubbele spaties
    text = re.sub(r"it\\'s", "it's", text)  # escaped apostrofs
    return text


# Reviews parsen
df_raw['reviews_list'] = df_raw['reviews'].apply(parse_list_column)
df_raw['notes_list']   = df_raw['notes'].apply(parse_list_column)

# Aantal reviews per parfum
df_raw['review_count'] = df_raw['reviews_list'].apply(len)

# Reviews samenvoegen tot één tekstveld
df_raw['reviews_text'] = df_raw['reviews_list'].apply(
    lambda lst: ' '.join([clean_text(r) for r in lst if len(r) > 20])
)

print(f'Parfums met minstens 1 review: {(df_raw["review_count"] > 0).sum():,}')
print(f'Gemiddeld aantal reviews per parfum: {df_raw["review_count"].mean():.1f}')
print()
print('Voorbeeld reviews_text:')
print(df_raw[df_raw['review_count'] > 0]['reviews_text'].iloc[0][:300])

In [ ]:
# Review lengte distributie
plt.figure(figsize=(8, 4))
df_raw['review_count'].clip(upper=20).plot(kind='hist', bins=20, color='steelblue', edgecolor='white')
plt.title('Aantal reviews per parfum (max 20 getoond)')
plt.xlabel('Aantal reviews')
plt.ylabel('Aantal parfums')
plt.tight_layout()
plt.show()

---
## 4. Datasets Samenvoegen

In [ ]:
def normalize_name(name):
    """Normaliseert parfumnamen voor matching."""
    if not isinstance(name, str):
        return ''
    name = name.lower().strip()
    name = re.sub(r'[^a-z0-9 ]', '', name)  # verwijder speciale tekens
    name = re.sub(r'\s+', ' ', name)
    return name


# Normaliseer namen
df_clean['_match_key'] = df_clean['Perfume'].apply(normalize_name)
df_raw['_match_key']   = df_raw['title'].apply(normalize_name)

# Merge
df_merged = df_clean.merge(
    df_raw[['_match_key', 'reviews_text', 'description', 'notes_list', 'review_count']],
    on='_match_key',
    how='left'
)

match_rate = df_merged['reviews_text'].notna().mean()
print(f'Totaal rijen na merge:  {len(df_merged):,}')
print(f'Match rate (reviews):   {match_rate:.1%}')
print(f'Parfums met reviews:    {df_merged["reviews_text"].notna().sum():,}')
print(f'Parfums zonder reviews: {df_merged["reviews_text"].isna().sum():,}')

---
## 5. Embedding Tekst Opbouwen

We combineren alle beschikbare tekstvelden tot één rijke `embedding_text` kolom.

In [ ]:
def build_embedding_text(row):
    """Bouwt een rijke tekst op voor embeddings per parfum."""
    parts = []
    
    # Basisinfo
    if pd.notna(row.get('Perfume')):
        parts.append(f"Perfume: {row['Perfume']}")
    if pd.notna(row.get('Brand')):
        parts.append(f"Brand: {row['Brand']}")
    if pd.notna(row.get('Gender')):
        parts.append(f"Gender: {row['Gender']}")
    if pd.notna(row.get('Year')):
        parts.append(f"Year: {row['Year']}")
    
    # Geur-noten
    for note_col in ['Top Notes', 'Middle Notes', 'Base Notes']:
        if pd.notna(row.get(note_col)) and str(row[note_col]).strip():
            parts.append(f"{note_col}: {row[note_col]}")
    
    # Accords
    accords = [str(row.get(f'Main Accord {i}', '')) 
               for i in range(1, 6) 
               if pd.notna(row.get(f'Main Accord {i}'))]
    if accords:
        parts.append(f"Main accords: {', '.join(accords)}")
    
    # Beschrijving uit raw dataset
    if pd.notna(row.get('description')) and str(row.get('description')).strip():
        parts.append(f"Description: {clean_text(str(row['description']))}")
    
    # Reviews
    if pd.notna(row.get('reviews_text')) and str(row.get('reviews_text')).strip():
        # Maximaal 500 tekens aan reviews meenemen
        rev_text = str(row['reviews_text'])[:500]
        parts.append(f"Reviews: {rev_text}")
    
    return ' | '.join(parts)


df_merged['embedding_text'] = df_merged.apply(build_embedding_text, axis=1)

print('Voorbeeld embedding_text:')
print('-' * 80)
print(df_merged['embedding_text'].iloc[0])

In [ ]:
# Tekst-lengte statistieken
df_merged['text_length'] = df_merged['embedding_text'].str.len()

print('Tekst-lengte statistieken (embedding_text):')
print(df_merged['text_length'].describe().round(0))

plt.figure(figsize=(8, 4))
df_merged['text_length'].clip(upper=2000).plot(kind='hist', bins=40, color='steelblue', edgecolor='white')
plt.title('Lengte van embedding_text per parfum')
plt.xlabel('Aantal tekens')
plt.ylabel('Aantal parfums')
plt.tight_layout()
plt.show()

---
## 6. Dataset Opslaan

In [ ]:
# Sla finale dataset op
df_merged.to_csv('fragrantica_final.csv', index=False)
print(f'Opgeslagen: fragrantica_final.csv ({len(df_merged):,} rijen)')
print()
print('Kolommen in finale dataset:')
print(df_merged.columns.tolist())

---
## 7. Embeddings Genereren (Sentence Transformers)

> ⚠️ Dit onderdeel vereist `sentence-transformers` en kan enkele minuten duren afhankelijk van de dataset-grootte.

In [ ]:
# Installeer indien nodig
# !pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Model laden (lichtgewicht multilinguaal model)
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model geladen:', model)

# Gebruik een subset voor testen (verwijder .head() voor volledige dataset)
df_sample = df_merged.dropna(subset=['embedding_text']).head(500).copy()

print(f'Embeddings genereren voor {len(df_sample):,} parfums...')
embeddings = model.encode(
    df_sample['embedding_text'].tolist(),
    batch_size=64,
    show_progress_bar=True
)

print(f'Embedding shape: {embeddings.shape}')

In [ ]:
# Embeddings opslaan voor hergebruik
import numpy as np
np.save('fragrantica_embeddings.npy', embeddings)
df_sample[['Perfume', 'Brand', 'Rating Value', 'embedding_text']].to_csv(
    'fragrantica_sample_index.csv', index=False
)
print('Embeddings opgeslagen: fragrantica_embeddings.npy')

---
## 8. Aanbevelingsfunctie (Proof of Concept)

In [ ]:
def recommend_perfumes(query: str, top_k: int = 5):
    """
    Geef aanbevelingen op basis van een vrije tekstquery.
    
    Parameters:
        query  : Beschrijving van gewenst parfum (bijv. 'fresh citrus summer')
        top_k  : Aantal aanbevelingen
    """
    # Encodeer de query
    query_embedding = model.encode([query])
    
    # Bereken cosine similarity met alle parfums
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    
    # Top-k indices
    top_indices = similarities.argsort()[::-1][:top_k]
    
    # Resultaten ophalen
    results = df_sample.iloc[top_indices][['Perfume', 'Brand', 'Rating Value', 'Gender']].copy()
    results['Similarity'] = similarities[top_indices].round(3)
    results = results.reset_index(drop=True)
    results.index += 1  # start bij 1
    
    return results


# Test
print('=== Aanbevelingen voor: "fresh citrus summer beach" ===')
recommend_perfumes('fresh citrus summer beach', top_k=5)

In [ ]:
# Nog een test
print('=== Aanbevelingen voor: "dark woody oud oriental" ===')
recommend_perfumes('dark woody oud oriental', top_k=5)

In [ ]:
# Interactieve query (zelf invoeren)
mijn_query = "sweet floral feminine rose jasmine"  # <-- aanpassen
print(f'=== Aanbevelingen voor: "{mijn_query}" ===')
recommend_perfumes(mijn_query, top_k=10)

---
## Samenvatting

| Stap | Output |
|------|--------|
| Data inladen | `df_clean`, `df_raw` |
| Reviews parsen | `reviews_text` kolom |
| Datasets mergen | `fragrantica_final.csv` |
| Embedding tekst | `embedding_text` kolom |
| Embeddings | `fragrantica_embeddings.npy` |
| PoC recommender | `recommend_perfumes()` functie |

**Volgende stap:** RAG-pipeline opzetten met een vector database (bijv. ChromaDB of FAISS)